In [1]:
suspect_cols = [
    "los_icu_days",           # FUTURE INFO - ICU stay length
    "death_30d",              # DIRECT OUTCOME LEAKAGE
    "death_48h",              # OUTCOME LEAKAGE  
    "stay_before_cvc_hours",  # Time-to-event info
    "stay_before_art_hours",  # Time-to-event info
    "icu_los_hours",          # FUTURE INFO
    "vent_start_in_icu",      # Event flag
    "stay_before_imv_hours",  # Time-to-event info
    "stay_before_iuc_hours",  # Time-to-event info
    "cvc_start_in_icu",       # Event flag
    "device_before_death"     # OUTCOME LEAKAGE
]

In [2]:
# ============================================
# SURVIVAL MODEL - PROPER FEATURE SELECTION
# ============================================

# These are the ONLY features allowed for survival prediction
BASELINE_FEATURES = [
    # Demographics (AT ADMISSION)
    "age",
    "gender",
    
    # ICU type (AT ADMISSION)
    "icu_micu",
    "icu_sicu", 
    "icu_ccu",
    "icu_neuro",
    "icu_trauma",
    
    # Comorbidities (HISTORICAL)
    "hypertension",
    "copd",
    "diabetes",
    "ckd",
    "chf",
    "stroke",
    "liver_disease",
    "cancer",
    
    # Severity scores (FIRST 24H)
    "apsiii",
    "sapsii",
    "sofa",
    "oasis",
    "lods",
    "gcs_min",
    
    # Vitals (FIRST 24H - AVERAGE)
    "heart_rate_mean",
    "mbp_mean",
    "sbp_mean",
    "dbp_mean",
    "resp_rate_mean",
    "temperature_mean",
    "spo2_mean",
    
    # Labs (FIRST 24H - AVERAGE)
    "wbc_mean",
    "platelets_mean",
    "hemoglobin_mean",
    "rbc_mean",
    "creatinine_mean",
    "bun_mean",
    "glucose_mean",
    "sodium_mean",
    "potassium_mean",
    "chloride_mean",
    "bicarbonate_mean",
    "aniongap_mean",
    "calcium_mean",
    "inr_mean",
    "pt_mean",
    "ptt_mean",
    "pao2fio2ratio_mean",
    "alt_mean",
    "alp_mean",
    "ast_mean",
    
    # Anthropometrics (FIRST DAY)
    "height",
    "weight",
    
    # Device exposure (BINARY - WHETHER USED, NOT WHEN)
    "imv",      # 0/1: invasive mech vent used?
    "cvc",      # 0/1: central line used?
    "iuc",      # 0/1: urinary catheter used?
]

# Features that should NEVER be included
FORBIDDEN_FEATURES = [
    # Outcomes
    "status", "survival_time", "deathtime", "death_time", 
    "death_30d", "death_48h", "hospital_expire_flag",
    
    # Temporal/future info
    "los_icu_days", "icu_los_hours", "los_hospital",
    "icu_intime", "icu_outtime", "admittime", "dischtime",
    
    # Time-to-event (gives away survival)
    "stay_before_imv_hours",
    "stay_before_cvc_hours",
    "stay_before_iuc_hours",
    "stay_before_art_hours",
    
    # Device timing (future information)
    "vent_start", "cvc_start", "iuc_start", "art_start",
    "index_device_time", "followup_end",
    "imv_risk_start", "cvc_risk_start", "iuc_risk_start",
    "risk_end",
    
    # Flags derived from outcomes
    "device_before_death",
    "vent_start_in_icu",
    "cvc_start_in_icu",
    
    # Infection outcomes
    "vap", "clabsi", "cauti",
    "device_infection", "device_infection_abx",
    "vap_abx", "clabsi_abx", "cauti_abx",
    
    # Identifiers
    "subject_id", "hadm_id", "stay_id",
]

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load your base cohort
cohort = pd.read_csv("mimic-iv-derived.csv")

# Parse datetime columns
datetime_cols = [
    "icu_intime", "icu_outtime", "admittime", "dischtime",
    "deathtime", "cvc_start", "art_start"
]
for c in datetime_cols:
    if c in cohort.columns:
        cohort[c] = pd.to_datetime(cohort[c], errors="coerce")

# =========================================
# ADD MISSING COMORBIDITIES FROM ICD CODES
# =========================================
DATA_DIR = Path("mimic-iv")

diagnoses = pd.read_csv(
    DATA_DIR / "diagnoses_icd.csv.gz",
    compression="gzip",
    usecols=["subject_id", "hadm_id", "icd_code", "icd_version"]
)

def make_comorbidity_flag(df, codes, name):
    """Extract comorbidity from ICD codes"""
    return (
        df.assign(flag=df["icd_code"].str.startswith(tuple(codes)))
          .groupby("hadm_id")["flag"]
          .max()
          .reset_index()
          .rename(columns={"flag": name})
    )

# ICD-9 and ICD-10 codes (paper-aligned)
htn_codes = ["401", "402", "403", "404", "405", "I10", "I11", "I12", "I13", "I15"]
copd_codes = ["491", "492", "496", "J44"]
dm_codes = ["250", "E10", "E11"]
ckd_codes = ["585", "N18"]
chf_codes = ["428", "I50"]
stroke_codes = ["430","431","432","433","434","435","436","437","438",
                "I60","I61","I62","I63","I64","G45","G46"]
liver_codes = ["570","571","572","573","K70","K71","K72","K73","K74","K75","K76"]
cancer_codes = ["140","141","142","143","144","145","146","147","148","149",
                "150","151","152","153","154","155","156","157","158","159",
                "160","161","162","163","164","165","166","167","168","169",
                "170","171","172","173","174","175","176","177","178","179",
                "180","181","182","183","184","185","186","187","188","189",
                "190","191","192","193","194","195","196","197","198","199",
                "C"]

# Extract comorbidities
htn = make_comorbidity_flag(diagnoses, htn_codes, "hypertension")
copd = make_comorbidity_flag(diagnoses, copd_codes, "copd")
diabetes = make_comorbidity_flag(diagnoses, dm_codes, "diabetes")
ckd = make_comorbidity_flag(diagnoses, ckd_codes, "ckd")
chf = make_comorbidity_flag(diagnoses, chf_codes, "chf")
stroke = make_comorbidity_flag(diagnoses, stroke_codes, "stroke")
liver = make_comorbidity_flag(diagnoses, liver_codes, "liver_disease")
cancer = make_comorbidity_flag(diagnoses, cancer_codes, "cancer")

# Merge into cohort
for df in [htn, copd, diabetes, ckd, chf, stroke, liver, cancer]:
    cohort = cohort.merge(df, on="hadm_id", how="left")

# Fill missing comorbidities as 0
comorb_cols = ["hypertension","copd","diabetes","ckd","chf","stroke","liver_disease","cancer"]
cohort[comorb_cols] = cohort[comorb_cols].fillna(0).astype(int)

# =========================================
# ADD ICU TYPE FLAGS
# =========================================
icustays = pd.read_csv(
    DATA_DIR / "icustays.csv.gz",
    compression="gzip",
    usecols=["stay_id", "first_careunit"]
)

cohort = cohort.merge(icustays, on="stay_id", how="left", validate="1:1")

# Create ICU type flags
cohort["icu_micu"] = cohort["first_careunit"].str.contains("Medical", na=False).astype(int)
cohort["icu_sicu"] = cohort["first_careunit"].str.contains("Surgical", na=False).astype(int)
cohort["icu_ccu"] = cohort["first_careunit"].str.contains("Coronary", na=False).astype(int)
cohort["icu_neuro"] = cohort["first_careunit"].str.contains("Neuro", na=False).astype(int)
cohort["icu_trauma"] = cohort["first_careunit"].str.contains("Trauma", na=False).astype(int)

# =========================================
# VERIFY ALL BASELINE FEATURES EXIST
# =========================================
missing_features = [f for f in BASELINE_FEATURES if f not in cohort.columns]
if missing_features:
    print(f"⚠️ WARNING: Missing features: {missing_features}")
    print("These must be added from MIMIC tables!")
else:
    print("✅ All baseline features present!")

⚠️ WARNING: Missing features: ['age', 'sapsii', 'wbc_mean', 'platelets_mean', 'hemoglobin_mean', 'rbc_mean', 'creatinine_mean', 'bun_mean', 'sodium_mean', 'potassium_mean', 'chloride_mean', 'bicarbonate_mean', 'aniongap_mean', 'calcium_mean', 'inr_mean', 'pt_mean', 'ptt_mean', 'pao2fio2ratio_mean', 'alt_mean', 'alp_mean', 'ast_mean', 'imv', 'cvc', 'iuc']
These must be added from MIMIC tables!


In [4]:
# =========================================
# 30-DAY SURVIVAL OUTCOMES (CORRECTED)
# =========================================

# Follow-up end = earliest of: discharge, death, or ICU+30 days
cohort["max_followup"] = cohort["icu_intime"] + pd.Timedelta(days=30)
cohort["followup_end"] = cohort[["dischtime", "deathtime", "max_followup"]].min(axis=1)

# Survival time in DAYS from ICU admission to follow-up end
cohort["survival_time"] = (
    (cohort["followup_end"] - cohort["icu_intime"]).dt.total_seconds() / (3600 * 24)
)

# Status = 1 if died within 30 days
cohort["status"] = (
    cohort["deathtime"].notna() &
    (cohort["survival_time"].notna()) &
    (cohort["survival_time"] <= 30)
).astype(int)

# Clip survival time at 30 days (right-censoring)
cohort["survival_time"] = cohort["survival_time"].clip(lower=0, upper=30)

print("30-day mortality rate:", cohort["status"].mean())
print("Median survival time (days):", cohort["survival_time"].median())

30-day mortality rate: 0.07520826183270413
Median survival time (days): 5.8902777777777775


In [6]:
"""
Diagnostic Script: Check Available MIMIC-IV Data
This will tell you exactly what files you have and what columns they contain
"""

import pandas as pd
from pathlib import Path

DATA_DIR = Path("mimic-iv")
DERIVED_CSV = "mimic-iv-derived.csv"

print("="*80)
print("MIMIC-IV DATA AVAILABILITY CHECK")
print("="*80)

# ============================================
# CHECK 1: What's in mimic-iv-derived.csv?
# ============================================
print("\n📋 CHECK 1: mimic-iv-derived.csv columns")
print("-"*80)

try:
    cohort = pd.read_csv(DERIVED_CSV, nrows=5)
    print(f"✅ File found: {cohort.shape[1]} columns, showing first 5 rows")
    print(f"\nAll columns ({len(cohort.columns)}):")
    for i, col in enumerate(cohort.columns, 1):
        print(f"  {i:2d}. {col}")
    
    # Check for specific important columns
    print(f"\n🔍 Checking for key columns:")
    key_cols = {
        "IDs": ["subject_id", "hadm_id", "stay_id"],
        "Demographics": ["age", "gender"],
        "Dates": ["icu_intime", "icu_outtime", "admittime", "dischtime", "deathtime"],
        "Severity": ["apsiii", "sapsii", "sofa", "oasis", "lods", "gcs_min"],
        "Vitals": ["heart_rate_mean", "mbp_mean", "sbp_mean", "resp_rate_mean", "temperature_mean", "spo2_mean"],
        "Labs": ["wbc_mean", "platelets_mean", "creatinine_mean", "bun_mean"],
        "Devices": ["imv", "cvc", "iuc"],
        "Comorbidities": ["hypertension", "copd", "diabetes", "ckd", "chf"],
        "ICU Type": ["icu_micu", "icu_sicu", "icu_ccu"]
    }
    
    for category, cols in key_cols.items():
        present = [c for c in cols if c in cohort.columns]
        missing = [c for c in cols if c not in cohort.columns]
        print(f"\n  {category}:")
        if present:
            print(f"    ✅ Present ({len(present)}): {', '.join(present[:5])}")
            if len(present) > 5:
                print(f"       ... and {len(present)-5} more")
        if missing:
            print(f"    ❌ Missing ({len(missing)}): {', '.join(missing[:5])}")
            if len(missing) > 5:
                print(f"       ... and {len(missing)-5} more")
    
except FileNotFoundError:
    print(f"❌ {DERIVED_CSV} not found!")
except Exception as e:
    print(f"❌ Error reading file: {e}")

# ============================================
# CHECK 2: What files are in mimic-iv/ directory?
# ============================================
print("\n\n📋 CHECK 2: Available MIMIC-IV raw tables")
print("-"*80)

if DATA_DIR.exists():
    # Look for common MIMIC-IV files
    common_files = [
        "admissions.csv.gz",
        "patients.csv.gz",
        "icustays.csv.gz",
        "diagnoses_icd.csv.gz",
        "procedures_icd.csv.gz",
        "procedureevents.csv.gz",
        "chartevents.csv.gz",
        "labevents.csv.gz",
        "microbiologyevents.csv.gz",
        "prescriptions.csv.gz",
        # Derived tables
        "complete_blood_count.csv.gz",
        "chemistry.csv.gz",
        "coagulation.csv.gz",
        "enzyme.csv.gz",
        "bg.csv.gz",
        "vitalsign.csv.gz",
        "first_day_height.csv.gz",
        "first_day_weight.csv.gz",
        "first_day_gcs.csv.gz",
        "first_day_sofa.csv.gz",
        "apsiii.csv.gz",
        "sapsii.csv.gz",
        "oasis.csv.gz",
        "lods.csv.gz",
    ]
    
    print("Checking for key files:")
    found_files = []
    missing_files = []
    
    for filename in common_files:
        filepath = DATA_DIR / filename
        if filepath.exists():
            # Get file size
            size_mb = filepath.stat().st_size / (1024 * 1024)
            found_files.append((filename, size_mb))
            print(f"  ✅ {filename:<35} ({size_mb:>8.1f} MB)")
        else:
            missing_files.append(filename)
    
    if missing_files:
        print(f"\n  ❌ Missing files ({len(missing_files)}):")
        for filename in missing_files[:10]:
            print(f"     - {filename}")
        if len(missing_files) > 10:
            print(f"     ... and {len(missing_files)-10} more")
    
    print(f"\n  Summary: {len(found_files)} files found, {len(missing_files)} missing")
    
else:
    print(f"❌ Directory '{DATA_DIR}' not found!")

# ============================================
# CHECK 3: Sample a few key tables
# ============================================
print("\n\n📋 CHECK 3: Sampling key tables")
print("-"*80)

tables_to_check = [
    ("patients.csv.gz", ["subject_id", "anchor_age", "gender"]),
    ("icustays.csv.gz", ["stay_id", "subject_id", "first_careunit"]),
    ("apsiii.csv.gz", ["stay_id", "apsiii"]),
    ("sapsii.csv.gz", ["stay_id", "sapsii"]),
]

for filename, expected_cols in tables_to_check:
    filepath = DATA_DIR / filename
    if filepath.exists():
        try:
            df = pd.read_csv(filepath, compression="gzip", nrows=3)
            print(f"\n  ✅ {filename}:")
            print(f"     Rows: {df.shape[0]} (showing first 3), Columns: {df.shape[1]}")
            print(f"     Columns: {', '.join(df.columns[:10])}")
            if len(df.columns) > 10:
                print(f"              ... and {len(df.columns)-10} more")
            
            # Check for expected columns
            present = [c for c in expected_cols if c in df.columns]
            missing = [c for c in expected_cols if c not in df.columns]
            if missing:
                print(f"     ⚠️ Missing expected: {', '.join(missing)}")
        except Exception as e:
            print(f"  ⚠️ {filename}: Error reading - {e}")
    else:
        print(f"  ❌ {filename}: Not found")

# ============================================
# CHECK 4: Recommendations
# ============================================
print("\n\n📋 CHECK 4: Recommendations")
print("-"*80)

print("\nBased on the checks above, here's what you should do:")
print("\n1. If mimic-iv-derived.csv is INCOMPLETE:")
print("   → Run the SQL queries from the paper's GitHub to create proper derived tables")
print("   → Or use the extract_missing_features.py script to fill in gaps")

print("\n2. If you're missing MIMIC-IV derived tables:")
print("   → Download them from PhysioNet (mimiciv/2.2/icu/ and mimiciv/2.2/hosp/)")
print("   → Or generate them using the MIMIC-Code repository:")
print("   → https://github.com/MIT-LCP/mimic-code/tree/main/mimic-iv/concepts")

print("\n3. If you have raw tables but missing derived:")
print("   → The paper's SQL uses BigQuery syntax")
print("   → You can adapt them to Python/pandas")
print("   → Or use DuckDB to run SQL on CSV files")

print("\n" + "="*80)
print("✅ DIAGNOSTIC CHECK COMPLETE")
print("="*80)

MIMIC-IV DATA AVAILABILITY CHECK

📋 CHECK 1: mimic-iv-derived.csv columns
--------------------------------------------------------------------------------
✅ File found: 57 columns, showing first 5 rows

All columns (57):
   1. stay_id
   2. subject_id
   3. hadm_id
   4. icu_intime
   5. icu_outtime
   6. los_icu_days
   7. admittime
   8. dischtime
   9. gender
  10. admission_age
  11. race
  12. hospital_expire_flag
  13. deathtime
  14. death_30d
  15. death_48h
  16. heart_rate_mean
  17. sbp_mean
  18. dbp_mean
  19. mbp_mean
  20. resp_rate_mean
  21. temperature_mean
  22. spo2_mean
  23. glucose_mean
  24. platelets_max
  25. wbc_max
  26. hemoglobin_max
  27. hematocrit_max
  28. creatinine_max
  29. bun_max
  30. sodium_max
  31. potassium_max
  32. chloride_max
  33. bicarbonate_max
  34. aniongap_max
  35. calcium_max
  36. bilirubin_max
  37. alt_max
  38. alp_max
  39. ast_max
  40. inr_max
  41. pt_max
  42. ptt_max
  43. pao2fio2ratio_max
  44. urineoutput
  45. height

In [8]:
from pathlib import Path
import pandas as pd

# Check what files exist
mimic_dir = Path(".")  # or wherever your MIMIC files are

# List all .csv.gz files
files = list(mimic_dir.glob("*.csv.gz"))
print(f"Found {len(files)} CSV files:")
for f in sorted(files):
    print(f"  - {f.name}")

Found 0 CSV files:


In [7]:
import pandas as pd

# Load your BigQuery-derived CSV
cohort = pd.read_csv("mimic-iv-derived.csv")

print(f"Shape: {cohort.shape}")
print(f"\nColumn names ({len(cohort.columns)} total):")
print(cohort.columns.tolist())

# Show first few rows
print(f"\nFirst 3 rows:")
print(cohort.head(3))

Shape: (64102, 57)

Column names (57 total):
['stay_id', 'subject_id', 'hadm_id', 'icu_intime', 'icu_outtime', 'los_icu_days', 'admittime', 'dischtime', 'gender', 'admission_age', 'race', 'hospital_expire_flag', 'deathtime', 'death_30d', 'death_48h', 'heart_rate_mean', 'sbp_mean', 'dbp_mean', 'mbp_mean', 'resp_rate_mean', 'temperature_mean', 'spo2_mean', 'glucose_mean', 'platelets_max', 'wbc_max', 'hemoglobin_max', 'hematocrit_max', 'creatinine_max', 'bun_max', 'sodium_max', 'potassium_max', 'chloride_max', 'bicarbonate_max', 'aniongap_max', 'calcium_max', 'bilirubin_max', 'alt_max', 'alp_max', 'ast_max', 'inr_max', 'pt_max', 'ptt_max', 'pao2fio2ratio_max', 'urineoutput', 'height', 'weight', 'apsiii', 'sofa', 'gcs_min', 'oasis', 'lods', 'rrt_flag', 'dialysis_type', 'cvc_start', 'art_start', 'stay_before_cvc_hours', 'stay_before_art_hours']

First 3 rows:
    stay_id  subject_id   hadm_id       icu_intime       icu_outtime  \
0  32398411    19850244  27972658   6/5/2162 15:25    6/9/216